# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


## 1. Two Paper Findings + My Methodology Questions

I'm picking two ML-appendix findings from the FlyRank paper to apply the same audit lens I'm about to turn on my own work. The paper is already upfront that these are exploratory — my goal here isn't to grade it, just to practice asking the same questions I'd want asked of my own notebook.

### Finding A — "What Predicts Health?" (Random Forest feature importance)

The paper runs a Random Forest to find which features predict Health Score, and reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors.

**Where does the label come from?**
Health Score is defined earlier in the paper as a direct formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). So the label is a weighted sum of the same features the model is being asked to "predict" from.

**Does the validation design carry the claim?**
Not fully — and the paper actually flags this itself, noting the importance is "descriptive rather than causal" since the target is partly constructed from these inputs. My question, asked constructively: if the top predictors are literally the ingredients of the label's formula, is the model teaching us anything new about content performance, or mostly reconstructing arithmetic it was already given? A cleaner test would be re-running importance on a target that isn't built from the candidate features.

### Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The paper reports 71% holdout accuracy for a logistic regression classifying growing vs. declining content, using an 80/20 split.

**Where does the label come from?**
The growth/decline label comes from the 30-day-vs-previous-30-day impression change (documented clearly in the paper's metric definitions). This part is clean — no ambiguity about label origin.

**Does the validation design carry the claim?**
Two open questions here. First, the dataset spans 57 brands, but the split described is a plain random 80/20 — there's no mention of grouping by brand, so it's unclear whether some brands' content appears in both train and test, which would let the model partly memorize brand-level patterns rather than learn general ones. Second, no base rate is reported alongside the 71% — without knowing what share of the sample is naturally "growing," it's hard to tell how much of that 71% is real signal versus what a naive majority-class guess would already get. Both are easy, low-cost additions (a `GroupKFold`-style split by brand, and printing the base rate) that would make the 71% number something a reader could actually act on.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [18]:
!git clone https://github.com/Ayesha-Shahzadkhan/flyrank-assignment1.git
%cd flyrank-assignment1
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

df.columns.tolist()

Cloning into 'flyrank-assignment1'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 168 (delta 74), reused 99 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.88 MiB | 9.55 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/flyrank-assignment1/flyrank-assignment1/flyrank-assignment1/flyrank-assignment1/flyrank-assignment1


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [19]:

df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
groups = df['client_id']

leak_cols = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d',
             'sessions_last_30d', 'impressions_prev_30d','clicks_prev_30d', 'sessions_prev_30d']
x = df.drop(columns=leak_cols + ['is_declining']).select_dtypes(include='number')
y = df['is_declining']

base_rate = y.mean()


In [20]:
x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(x, y, stratify=y, test_size=0.1, random_state=42)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x, y, groups=groups))

x_train_g = x.iloc[train_idx]
x_test_g = x.iloc[test_idx]
y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

overlap = set(df.loc[x_train_g.index, 'client_id']) & set(df.loc[x_test_g.index, 'client_id'])
print(len(overlap))

0


In [21]:
def evaluate(x_tr, x_te, y_tr, y_te, label):
    # Impute missing values with the median of the training data
    for col in x_tr.columns:
        if x_tr[col].isnull().any():
            median_val = x_tr[col].median()
            x_tr[col] = x_tr[col].fillna(median_val)
            x_te[col] = x_te[col].fillna(median_val)

    model = LogisticRegression(max_iter=1000)
    model.fit(x_tr, y_tr)
    pred = model.predict(x_te) # Assign predictions to 'pred'
    return {
        'split': label,
        'base_rate': y_te.mean(),
        'accuracy': accuracy_score(y_te, pred),
        'precision': precision_score(y_te, pred),
        'recall': recall_score(y_te, pred),
        'f1': f1_score(y_te, pred),
    }

results = pd.DataFrame([
    evaluate(x_train_r.copy(), x_test_r.copy(), y_train_r.copy(), y_test_r.copy(), 'Random split'),
    evaluate(x_train_g.copy(), x_test_g.copy(), y_train_g.copy(), y_test_g.copy(), 'Grouped split'),
])

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

Doing the same leakage check from Week 3, but on the feature set I'm actually using now.
`trend_direction` and `trend_pct` are already out since the label comes straight from them.
The columns I hadn't stress-tested yet are the 90-day totals — impressions_90d, sessions_90d,
clicks_90d, and a few others — because they numerically overlap with the same last-30d and
prev-30d windows my label is built from. If the model quietly leans on that overlap, removing
these columns should hurt it.

In [24]:
suspect_cols = ['impressions_90d', 'sessions_90d', 'clicks_90d',
                 'pageviews_90d', 'users_90d', 'engaged_sessions_90d']

X_no_suspect = x.drop(columns=suspect_cols)

x_train_g_ns = X_no_suspect.iloc[train_idx]
x_test_g_ns  = X_no_suspect.iloc[test_idx]

leakage_check = pd.DataFrame([
    evaluate(x_train_g.copy(), x_test_g.copy(), y_train_g.copy(), y_test_g.copy(), 'Grouped split — WITH suspects'),
    evaluate(x_train_g_ns, x_test_g_ns, y_train_g.copy(), y_test_g.copy(), 'Grouped split — WITHOUT suspects'),
])
leakage_check

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/tmp/ipykernel_516/3330571724.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_tr[col] = x_tr[col].fillna(median_val)
/tmp/ipykernel_516/3330571724.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

,split,base_rate,accuracy,precision,recall,f1
0,Grouped split — WITH suspects,0.510952,0.582671,0.592143,0.588758,0.590446
1,Grouped split — WITHOUT suspects,0.510952,0.577803,0.587576,0.582725,0.585140


With the suspect columns in, accuracy was **0.583**; without them, **0.578**.

The score barely moved (less than a 1-point drop), so these 90-day totals don't seem to be
giving the model a shortcut. The overlap with the label's 30-day window is probably too small
relative to the full 90-day total to matter — in other words, the aggregate is diluted enough
that it isn't functioning as a hidden answer key. I'm keeping these columns in the feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

Here's the boldest line from my Week-5 notebook:

> "Logistic Regression edges out the baseline on accuracy and precision, but the baseline
> still recalls more actual decline cases (0.822 vs 0.657)."

And here's a more honest version of it:

Logistic Regression showed higher accuracy and precision than the rule-based baseline on this
dataset under a random split, while the baseline had better recall. Once I switched to a
grouped-by-client split — a fairer test — the gap between the two came out to **[state the
observed gap]**, which I'd call a directional result, not a settled one, since it's based on
one dataset and one split. Neither model is ready to make decisions on its own yet; both work
better as a way to flag content for someone to manually review, not as an automatic call.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.